<a href="https://colab.research.google.com/github/harshal8704/GenAi-Practicals/blob/main/GenAI_Prac5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Practical 5: Transfer Learning for Text Classification

In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [2]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer

# Labeled news dataset
dataset_dict = {
    "text": [
        "The team won the championship match after scoring a late goal in extra time.",
        "The striker signed a multi-million dollar contract extension today.",
        "The basketball team secured a playoff spot with a last-second buzzer-beater.",
        "The tennis legend announced retirement after winning twenty Grand Slams.",
        "The prime minister proposed a new tax policy bill in parliament today.",
        "Election results confirmed a massive voter turnout across major districts.",
        "Diplomats met for bilateral trade talks to lower import tariff rates.",
        "The senate passed the new healthcare reform bill after midnight debate.",
        "The tech company launched a new smartphone with an integrated AI chip.",
        "Engineers unveiled a breakthrough in high-density quantum computing architecture.",
        "Software developers released an urgent security patch for cloud infrastructure.",
        "Researchers trained a new transformer neural network model for translation."
    ],
    "label": [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2] # 0: Sports, 1: Politics, 2: Tech
}

raw_dataset = Dataset.from_dict(dataset_dict)
split_dataset = raw_dataset.train_test_split(test_size=0.25, seed=42)

# Load tokenizer
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [3]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "Sports", 1: "Politics", 2: "Technology"}
label2id = {"Sports": 0, "Politics": 1, "Technology": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    macro_f1 = f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": macro_f1}

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.206060,0.000000,0.000000
2,1.015632,1.203097,0.000000,0.000000
3,1.015632,1.189200,0.000000,0.000000
4,1.011677,1.177171,0.000000,0.000000
5,0.897324,1.174171,0.333333,0.333333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=0.9748778661092122, metrics={'train_runtime': 29.8409, 'train_samples_per_second': 1.508, 'train_steps_per_second': 0.503, 'total_flos': 1490284811520.0, 'train_loss': 0.9748778661092122, 'epoch': 5.0})

In [5]:
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

unseen_articles = [
    "The quarterback threw three touchdown passes to seal the championship.",
    "Parliament members voted on the proposed national infrastructure bill.",
    "A novel GPU microchip design boosts deep learning inference speed threefold."
]

predictions = classifier(unseen_articles)

for text, pred in zip(unseen_articles, predictions):
    print(f"Article: {text}")
    print(f"Predicted Category: {pred['label']} | Confidence: {pred['score']:.4f}\n")

Article: The quarterback threw three touchdown passes to seal the championship.
Predicted Category: Sports | Confidence: 0.4145

Article: Parliament members voted on the proposed national infrastructure bill.
Predicted Category: Technology | Confidence: 0.3598

Article: A novel GPU microchip design boosts deep learning inference speed threefold.
Predicted Category: Technology | Confidence: 0.4530

